# Graph Classification with TopKPooling on Colors Dataset

Graph Classification on TUDataset (Colors): this notebook ports the PyG "attention supervision" example ("Understanding Attention and Generalization in Graph Neural Networks", https://arxiv.org/abs/1905.02850) to K3-Node. COLORS-3 encodes a per-node importance signal in the first feature column; `HandleNodeAttention` turns it into a per-graph softmax target `data.attn` and drops it from the input features. `K3Net` scores `TopKPooling` on the *raw* node features (supervised against `data.attn` via a KL-divergence auxiliary loss), while the actual pooled representation comes from a `GINConv` embedding — so the network is pushed to make its pooling attention match the known-good attention pattern. The single code cell installs **K3-Node**, builds the 500/2500/7500 train/val/test split, and runs a manual multi-backend training loop (needed because the loss combines a regression term with the per-graph KL term, and accuracy/pooling-ratio are reported for train/val/test every epoch) — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import copy
import numpy as np
import keras
from keras import layers, ops

from k3_node import layers as k3_layers
from k3_node.layers.conv.utils import scatter
from k3_node.datasets import TUDataset
from k3_node.loader import DataLoader

title = "Graph Classification with TopKPooling on Colors Dataset"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")


# 1. Attention-supervision transform: COLORS-3 stores a per-node importance
# signal in feature column 0. Turn it into a per-graph softmax target and
# drop it from the input features.
class HandleNodeAttention:
    def __call__(self, data):
        data = copy.copy(data)
        data.attn = ops.softmax(data.x[:, 0], axis=0)
        data.x = data.x[:, 1:]
        return data


dataset = TUDataset(root="./data/COLORS-3", name="COLORS-3", use_node_attr=True,
                    transform=HandleNodeAttention())

train_loader = DataLoader(dataset[:500], batch_size=60, shuffle=True)
val_loader = DataLoader(dataset[500:3000], batch_size=60)
test_loader = DataLoader(dataset[3000:], batch_size=60)

in_channels = dataset.num_features


# 2. GINConv + attention-supervised TopKPooling
class K3Net(keras.Model):
    def __init__(self, in_channels, hidden_channels=64):
        super().__init__()
        self.mlp1 = keras.Sequential([layers.Dense(hidden_channels, activation="relu"), layers.Dense(hidden_channels)])
        self.conv1 = k3_layers.GINConv(self.mlp1)
        # Scores nodes using the *raw* input features (attn=x below), not the
        # conv1 embedding -- matches the reference exactly.
        self.pool1 = k3_layers.TopKPooling(in_channels, min_score=0.05)
        self.mlp2 = keras.Sequential([layers.Dense(hidden_channels, activation="relu"), layers.Dense(hidden_channels)])
        self.conv2 = k3_layers.GINConv(self.mlp2)
        self.lin = layers.Dense(1)

    def call(self, x, edge_index, batch, attn_target):
        out = ops.relu(self.conv1(x, edge_index))
        out, edge_index, _, batch, perm, score = self.pool1(out, edge_index, batch=batch, attn=x)
        ratio = ops.cast(ops.shape(out)[0], "float32") / ops.cast(ops.shape(x)[0], "float32")

        out = ops.relu(self.conv2(out, edge_index))
        out = k3_layers.global_add_pool(out, batch)
        out = ops.reshape(self.lin(out), (-1,))

        # KL(target || score), elementwise then per-graph mean -- matches
        # `F.kl_div(log(score + eps), attn[perm], reduction='none')` followed
        # by `scatter(..., reduce='mean')` in the reference.
        target = ops.take(attn_target, perm, axis=0)
        kl_term = target * (ops.log(target + 1e-14) - ops.log(score + 1e-14))
        attn_loss = scatter(kl_term, batch, reduce="mean")

        return out, attn_loss, ratio


model = K3Net(in_channels)

# Eager forward pass to build every sublayer's weights
sample = next(iter(train_loader))
_ = model(
    ops.convert_to_tensor(sample.x, dtype="float32"),
    ops.convert_to_tensor(sample.edge_index, dtype="int64"),
    ops.convert_to_tensor(sample.batch, dtype="int64"),
    ops.convert_to_tensor(sample.attn, dtype="float32"),
)

optimizer = keras.optimizers.Adam(learning_rate=0.001)
trainable_vars = model.trainable_variables

# Setup optimizer variables for the functional (JAX) backend
if backend == "jax":
    import jax
    optimizer.build(trainable_vars)
    opt_vars = [v.value for v in optimizer.variables]
    non_trainable_vars = [v.value for v in model.non_trainable_variables]


def batch_tensors(data_batch):
    x = ops.convert_to_tensor(data_batch.x, dtype="float32")
    edge_index = ops.convert_to_tensor(data_batch.edge_index, dtype="int64")
    batch_vec = ops.convert_to_tensor(data_batch.batch, dtype="int64")
    attn = ops.convert_to_tensor(data_batch.attn, dtype="float32")
    y = ops.reshape(ops.convert_to_tensor(data_batch.y, dtype="float32"), (-1,))
    return x, edge_index, batch_vec, attn, y


def loss_fn(out, attn_loss, y):
    return ops.mean(ops.square(out - y) + 100 * attn_loss)


# 3. Multi-Backend Training Step
def train_step(data_batch):
    global opt_vars
    x, edge_index, batch_vec, attn, y = batch_tensors(data_batch)
    num_graphs = int(ops.shape(y)[0])

    if backend == "torch":
        out, attn_loss, _ = model(x, edge_index, batch_vec, attn)
        loss = loss_fn(out, attn_loss, y)
        loss.backward()
        grads = [v.value.grad for v in trainable_vars]
        optimizer.apply_gradients(zip(grads, trainable_vars))
        for v in trainable_vars:
            if v.value.grad is not None:
                v.value.grad.zero_()
        loss_value = float(ops.convert_to_numpy(loss))

    elif backend == "tensorflow":
        import tensorflow as tf
        with tf.GradientTape() as tape:
            out, attn_loss, _ = model(x, edge_index, batch_vec, attn)
            loss = loss_fn(out, attn_loss, y)
        grads = tape.gradient(loss, trainable_vars)
        optimizer.apply_gradients(zip(grads, trainable_vars))
        loss_value = float(ops.convert_to_numpy(loss))

    else:  # jax
        trainable_values = [v.value for v in trainable_vars]

        def _loss_fn(params):
            (out, attn_loss, _), _ = model.stateless_call(params, non_trainable_vars, x, edge_index, batch_vec, attn)
            return loss_fn(out, attn_loss, y)

        loss_val, grads = jax.value_and_grad(_loss_fn)(trainable_values)
        new_values, opt_vars = optimizer.stateless_apply(opt_vars, grads, trainable_values)
        for v, val in zip(trainable_vars, new_values):
            v.assign(val)
        loss_value = float(loss_val)

    return loss_value * num_graphs


def train():
    total_loss = 0.0
    for data_batch in train_loader:
        total_loss += train_step(data_batch)
    return total_loss / len(train_loader.dataset)


def test(loader):
    corrects, total_ratio, num_batches = [], 0.0, 0
    for data_batch in loader:
        x, edge_index, batch_vec, attn, y = batch_tensors(data_batch)
        out, _, ratio = model(x, edge_index, batch_vec, attn)
        pred = ops.cast(ops.round(out), "int64")
        corrects.append(ops.convert_to_numpy(pred == ops.cast(y, "int64")))
        total_ratio += float(ops.convert_to_numpy(ratio))
        num_batches += 1
    return np.concatenate(corrects, axis=0), total_ratio / num_batches


print(f"Training K3-Node attention-supervised TopKPooling model on {backend} backend...")
for epoch in range(1, 301):
    loss = train()
    train_correct, train_ratio = test(train_loader)
    val_correct, val_ratio = test(val_loader)
    test_correct, test_ratio = test(test_loader)

    train_acc = train_correct.sum() / train_correct.shape[0]
    val_acc = val_correct.sum() / val_correct.shape[0]

    test_acc1 = test_correct[:2500].sum() / 2500
    test_acc2 = test_correct[2500:5000].sum() / 2500
    test_acc3 = test_correct[5000:].sum() / 2500

    print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.3f}, "
          f"Val: {val_acc:.3f}, Test Orig: {test_acc1:.3f}, "
          f"Test Large: {test_acc2:.3f}, Test LargeC: {test_acc3:.3f}, "
          f"Train/Val/Test Ratio="
          f"{train_ratio:.3f}/{val_ratio:.3f}/{test_ratio:.3f}")

print("\n✓ K3-Node execution completed successfully!")